# Cell Subtyping (I) data prep

This workflow prepares the data for cell typing using MapMyCells.

Input file used here was produced in 'FilteredDataset_02_data_filtering.ipyb'

## 1. Package Imports and Configuration

In [ ]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import os
from pathlib import Path

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime


## 2. Data structure

Define paths and input files

In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

#### Folder to process:
Modifiy the object below accordingly.

In [ ]:
# Brain region folder to process
region = "substantia_nigra"


In [ ]:

# Define a local path for workshop files
local_data_path = WS_FILES / region

# map my cells directories
mapmycells_input_dir = ( local_data_path / "mapmycells/input" )
mapmycells_output_dir = ( local_data_path / "mapmycells/output" )

# other directories
input_path =( local_data_path / "input")
output_path =( local_data_path / "output")

# Make sure the directories exists
os.makedirs(mapmycells_input_dir, exist_ok=True)
os.makedirs(mapmycells_output_dir, exist_ok=True)
os.makedirs(input_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    print("Local folder missing")

print(f"Local data directory ready at: {local_data_path}")

Input data (full gene dataset) is generated in workflow 'FilteredDataset_02__data_filtering.ipynb'

In [ ]:
# Load curated AnnData object (full gene space)
adata_path = input_path / f"asap-cohort.{region}_cells_full_genes_curated.h5ad"

adata = sc.read_h5ad(adata_path)

## 3. Data processing and prepare data for export

In [ ]:
# Preserve raw counts and original expression matrix
adata.layers['counts'] = adata.X.copy()

### 3.1. Remove bulky objects from AnnData
In order to improve efficiency, remove some heavy object from the adata object that is not needed

In [ ]:
# Drop large neighbor matrices
for k in ['neighbors_scvi_distances','neighbors_scvi_connectivities']:
    adata.obsp.pop(k, None)


# Drop heavy per-gene stats (keep only highly_variable + n_cells)
adata.var.drop(columns=[
    'means','variances','residual_variances',
    'highly_variable_rank','highly_variable_nbatches','highly_variable_intersection'
], errors='ignore', inplace=True)

### 3.2. Make Compatible MapMyCells
Format data to match the reference mapping requirements.

MapMyCells expects gene features as ENSG-style Ensembl IDs in the row names, rather than gene symbols.

In [ ]:
# Preserve gene_name
adata.var["gene_name"] = adata.var_names

# Set var_names to ENSG IDs
adata.var_names = adata.var["gene_id"].astype(str)

# Ensure uniqueness
adata.var_names_make_unique()

# Quick check
print(adata.var.head())
print(adata.var_names[:5])


## 4. Save file

In [ ]:
processed_filename = input_path / f"asap-cohort_{region}_cells_preprocessed.h5ad"

adata.write_h5ad(processed_filename)